# AI Data Poisoning Mitigation - Colab Setup

This notebook sets up GPT-2 medium on Google Colab with GPU.

**Steps:**
1. Clone your repo
2. Install dependencies
3. Mount Google Drive (for checkpoint persistence)
4. Download and verify GPT-2 medium
5. Run verification tests

**Hardware:** Set Runtime → Change runtime type → GPU (T4 recommended)

## Step 1: Clone Repository

In [ ]:
# Clone your Mini-Project repo
!git clone https://github.com/Jinendran10/Mini-Project.git
%cd Mini-Project
!pwd

## Step 2: Install Dependencies

This installs PyTorch, Transformers, FAISS, and all required packages.

In [ ]:
# Install all dependencies from requirements.txt
!pip install -q -r requirements.txt

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## Step 3: Mount Google Drive (for Checkpoints)

This lets you save model checkpoints to Drive so they persist after the session ends.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/mitigation_checkpoints'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {drive_checkpoint_dir}")

# Update config to use Drive for checkpoints
import yaml
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['data']['checkpoint_dir'] = drive_checkpoint_dir
config['model']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Config updated to use Google Drive for checkpoints")

## Step 4: Download GPT-2 Medium

Downloads gpt2-medium (355M parameters) and saves it locally.

In [ ]:
# Download gpt2-medium and save to checkpoints
!python scripts/download_model.py --model gpt2-medium --test

## Step 5: Verify Model Setup

Tests that the model can:
- Extract hidden states (for RLOD)
- Compute gradients (for TracIn)
- Save/load checkpoints

In [ ]:
# Run full verification with gpt2-medium
!python scripts/verify_model_setup.py

## Step 6: Run TracIn Influence Demo

Demonstrates how to compute influence scores between samples.

In [ ]:
# Run TracIn demo
!python scripts/verify_model_setup.py --demo-tracin

## Step 7: Test Model Access Patterns

Shows the 3 code patterns you'll use in JIE/RLOD/training.

In [ ]:
# Test the 3 key access patterns
!python scripts/test_model_access.py

## Optional: Manual Model Access Test

Run Python code directly in the notebook to test model access.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load gpt2-medium
model_name = "gpt2-medium"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Quick test
text = "The future of AI safety requires"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    embeddings = outputs.hidden_states[-1]
    print(f"✓ Hidden states shape: {embeddings.shape}")
    print(f"✓ Model ready for mitigation system")

## Step 8: Save Work to Drive and Download

Copy important files to Drive so you can access them later.

In [ ]:
import shutil
from pathlib import Path

# Copy checkpoints to Drive
local_checkpoints = Path('checkpoints')
if local_checkpoints.exists():
    shutil.copytree(local_checkpoints, f"{drive_checkpoint_dir}/model_files", dirs_exist_ok=True)
    print(f"✓ Checkpoints copied to {drive_checkpoint_dir}/model_files")

# List what's saved
!ls -lh {drive_checkpoint_dir}

print("\n✓ Setup complete! Your model files are saved to Google Drive.")
print("\nNext steps:")
print("1. Open this notebook in VS Code (File → Open)")
print("2. Start building JIE (TracIn) module")
print("3. Start building RLOD (outlier detection) module")

---

## Summary

**What you did:**
- ✓ Cloned repo to Colab
- ✓ Installed dependencies (PyTorch, Transformers, FAISS)
- ✓ Mounted Google Drive for persistence
- ✓ Downloaded GPT-2 medium (355M params)
- ✓ Verified model access (embeddings, gradients, checkpoints)
- ✓ Saved checkpoints to Drive

**Model capabilities confirmed:**
- Hidden states → RLOD can use for kNN outlier detection
- Gradients → TracIn can compute influence scores
- Checkpointing → Training loop can track progress

**Next: Build mitigation modules**
- `src/jie/tracin.py` - Influence estimation
- `src/rlod/detector.py` - Outlier detection  
- `src/robust_training/trainer.py` - Robust training pipeline